# LIDC-IDRI Deep Learning Pretraining

Pretrain a **3D ResNet-18** encoder on the **LIDC-IDRI** dataset (1,009 patients) using a
**segmentation** proxy task. The learned representations will then be used to extract
deep-feature vectors for the NSCLC-Radiomics cohort.

| Stage | Description |
|-------|-------------|
| **Data discovery** | Scan NIfTI directory, merge per-annotator masks into consensus |
| **Dataset & DataLoader** | Custom PyTorch Dataset with patch extraction + augmentation |
| **Model** | ResNet3D-18 encoder + SegmentationDecoder, binary (background/nodule) |
| **Training loop** | Segmentation pretraining with early stopping, MPS acceleration |
| **Validation** | Track Dice score + CE loss per epoch |
| **Checkpoint** | Save encoder weights for downstream feature extraction |

## 1. Setup & Dependencies

In [ ]:
import os, sys, logging, warnings, time, json, gc
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import SimpleITK as sitk

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

# Pipeline imports
sys.path.insert(0, str(Path.cwd()))
from radiomics_pipeline.config import PipelineConfig, PretrainingConfig
from radiomics_pipeline.preprocessing import ImagePreprocessor
from radiomics_pipeline.models.encoder import create_encoder, ResNet3D, BasicBlock
from radiomics_pipeline.models.pretraining import SegmentationDecoder

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(name)s %(levelname)s %(message)s")
logger = logging.getLogger("LIDC_Pretrain")

# MPS does not support 3D convolutions — must use CPU for this model
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"PyTorch {torch.__version__}")
print(f"Device:  {DEVICE}")
print(f"SimpleITK {sitk.Version.VersionString()}")

## 2. Data Discovery: LIDC-IDRI NIfTI

Scan the SMB-mounted LIDC directory. Each patient folder has:
- `ct.nii.gz` — the CT scan
- `Segmentation_of_Nodule_N_-_Annotation_*.nii.gz` — per-annotator masks

We merge all annotator masks for all nodules into a single **consensus binary mask**
(≥ 1 annotator = foreground). Patients without segmentations are skipped.

In [ ]:
LIDC_ROOT = Path("/Volumes/AIR-NSCLC/nifti_output/LIDC-IDRI")
assert LIDC_ROOT.exists(), f"LIDC root not found: {LIDC_ROOT}"

def discover_lidc_patients(root: Path) -> List[Dict]:
    """Scan LIDC directory and catalogue CT + segmentation files per patient."""
    patients = []
    for patient_dir in sorted(root.iterdir()):
        if not patient_dir.is_dir():
            continue
        ct_path = patient_dir / "ct.nii.gz"
        if not ct_path.exists():
            continue
        # Find all segmentation NIfTIs
        seg_files = sorted(patient_dir.glob("Segmentation_of_Nodule_*.nii.gz"))
        if len(seg_files) == 0:
            continue  # skip patients without annotations
        patients.append({
            "patient_id": patient_dir.name,
            "ct_path": ct_path,
            "seg_paths": seg_files,
            "n_seg_files": len(seg_files),
        })
    return patients

print("Scanning LIDC-IDRI directory (this may take a minute over SMB)...")
lidc_patients = discover_lidc_patients(LIDC_ROOT)
print(f"\n✓ Found {len(lidc_patients)} patients with CT + segmentation masks")
print(f"  Seg files per patient: min={min(p['n_seg_files'] for p in lidc_patients)}, "
      f"max={max(p['n_seg_files'] for p in lidc_patients)}, "
      f"median={np.median([p['n_seg_files'] for p in lidc_patients]):.0f}")
print(f"  First: {lidc_patients[0]['patient_id']}  Last: {lidc_patients[-1]['patient_id']}")

## 3. Pipeline & Pretraining Configuration

In [ ]:
# --- Pipeline config (image preprocessing) --------------------------------
pipeline_config = PipelineConfig(
    target_spacing=(1.0, 1.0, 1.0),
    hu_min=-1000.0,
    hu_max=400.0,
    min_mask_volume_mm3=10.0,      # LIDC nodules can be small
    max_mask_volume_mm3=500_000.0,
)

# --- Pretraining config ----------------------------------------------------
PATCH_SIZE   = (64, 64, 64)        # 3D patch dims
BATCH_SIZE   = 4                   # fits ~6 GB on MPS
NUM_EPOCHS   = 30                  # increase for full run
LEARNING_RATE = 1e-4
LATENT_DIM   = 512
VAL_FRACTION = 0.15                # 15 % validation split
ENCODER_TYPE = "resnet3d_18"       # lighter than ResNet-50
PATCHES_PER_PATIENT = 8            # pre-extracted patches per patient

CHECKPOINT_DIR = Path("/Users/danieljones/Documents/Cortex-Vault/Projects/Academic/University of Edinburgh/MSc Thesis/04. Codebase/Tests/checkpoints/lidc_pretrain")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Pre-extracted patches stored on the SMB share (data governance)
PATCH_CACHE_DIR = LIDC_ROOT.parent / "LIDC-IDRI_patches_64"

print(f"Encoder:     {ENCODER_TYPE}")
print(f"Latent dim:  {LATENT_DIM}")
print(f"Patch size:  {PATCH_SIZE}")
print(f"Batch size:  {BATCH_SIZE}")
print(f"Epochs:      {NUM_EPOCHS}")
print(f"LR:          {LEARNING_RATE}")
print(f"Patches/pt:  {PATCHES_PER_PATIENT}")
print(f"Patch cache: {PATCH_CACHE_DIR}")
print(f"Checkpoint:  {CHECKPOINT_DIR}")

## 4. Train / Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

train_patients, val_patients = train_test_split(
    lidc_patients, test_size=VAL_FRACTION, random_state=42
)

print(f"Train patients: {len(train_patients)}")
print(f"Val patients:   {len(val_patients)}")

## 5. LIDC Dataset: Consensus Mask Merging & Patch Extraction

Custom PyTorch `Dataset` that:
1. Loads the CT scan and all per-annotator segmentation NIfTIs
2. Merges them into a **binary consensus mask** (union of all annotations)
3. Preprocesses (resample to 1mm isotropic, clamp HU, z-score normalise)
4. Extracts a random 64³ patch (70% centred on foreground, 30% random)

In [ ]:
class LIDCPretrainingDataset(Dataset):
    """LIDC-IDRI dataset for segmentation-based encoder pretraining."""

    def __init__(
        self,
        patients: List[Dict],
        pipeline_config: PipelineConfig,
        patch_size: Tuple[int, int, int] = (64, 64, 64),
        augment: bool = True,
    ):
        self.patients = patients
        self.preprocessor = ImagePreprocessor(pipeline_config)
        self.patch_size = patch_size
        self.augment = augment

    def __len__(self) -> int:
        return len(self.patients)

    def __getitem__(self, idx: int) -> Dict:
        rec = self.patients[idx]
        ct_image = sitk.ReadImage(str(rec["ct_path"]))

        # --- Merge annotator masks into a single binary mask ----------------
        ref_image = ct_image  # use CT geometry as reference
        merged_arr = None
        for seg_path in rec["seg_paths"]:
            try:
                seg = sitk.ReadImage(str(seg_path))
                # Handle 4-D segmentation volumes (e.g. DICOM SEG with extra dim)
                if seg.GetDimension() > 3:
                    arr_4d = sitk.GetArrayFromImage(seg)
                    # Collapse extra dims → 3-D by max-projection along leading axes
                    while arr_4d.ndim > 3:
                        arr_4d = arr_4d.max(axis=0)
                    seg = sitk.GetImageFromArray(arr_4d)
                    seg.SetOrigin(ref_image.GetOrigin())
                    seg.SetSpacing(ref_image.GetSpacing())
                    seg.SetDirection(ref_image.GetDirection())
                # Resample seg to CT space if grids differ
                if seg.GetSize() != ref_image.GetSize() or \
                   seg.GetSpacing() != ref_image.GetSpacing():
                    seg = sitk.Resample(
                        seg, ref_image,
                        sitk.Transform(),
                        sitk.sitkNearestNeighbor, 0,
                        seg.GetPixelID(),
                    )
                arr = sitk.GetArrayFromImage(seg)
                if merged_arr is None:
                    merged_arr = (arr > 0).astype(np.uint8)
                else:
                    merged_arr = np.maximum(merged_arr, (arr > 0).astype(np.uint8))
            except Exception as e:
                logger.warning(f"Skipping seg {seg_path.name}: {e}")
                continue

        # Fallback: if all segs failed, use an empty mask
        if merged_arr is None:
            merged_arr = np.zeros(sitk.GetArrayFromImage(ct_image).shape, dtype=np.uint8)

        # Binarise and convert back to SimpleITK for preprocessing
        mask_image = sitk.GetImageFromArray(merged_arr.astype(np.int32))
        mask_image.CopyInformation(ref_image)

        # --- Preprocess (resample, clamp HU, z-score) ----------------------
        image_arr, mask_arr = self.preprocessor.preprocess_for_deep_learning(
            ct_image, mask_image
        )
        # image_arr: (1, D, H, W) float32;  mask_arr: (D, H, W) int64

        # --- Extract random patch ------------------------------------------
        image_patch, mask_patch = self._extract_patch(image_arr, mask_arr)

        # --- Augmentation (random flip + small intensity jitter) -----------
        if self.augment:
            image_patch, mask_patch = self._augment(image_patch, mask_patch)

        return {
            "image": torch.from_numpy(image_patch).float(),
            "mask":  torch.from_numpy(mask_patch).long(),
            "patient_id": rec["patient_id"],
        }

    # --- helpers -----------------------------------------------------------

    def _extract_patch(self, image: np.ndarray, mask: np.ndarray):
        """Random 3-D patch, 70 % foreground-centred."""
        _, d, h, w = image.shape
        pd, ph, pw = self.patch_size
        # Clamp patch size to image dims
        pd, ph, pw = min(pd, d), min(ph, h), min(pw, w)

        fg = np.where(mask > 0)
        if len(fg[0]) > 0 and np.random.random() < 0.7:
            i = np.random.randint(len(fg[0]))
            cd, ch, cw = fg[0][i], fg[1][i], fg[2][i]
            sd = max(0, min(cd - pd // 2, d - pd))
            sh = max(0, min(ch - ph // 2, h - ph))
            sw = max(0, min(cw - pw // 2, w - pw))
        else:
            sd = np.random.randint(0, max(1, d - pd + 1))
            sh = np.random.randint(0, max(1, h - ph + 1))
            sw = np.random.randint(0, max(1, w - pw + 1))

        ip = image[:, sd:sd+pd, sh:sh+ph, sw:sw+pw]
        mp = mask[sd:sd+pd, sh:sh+ph, sw:sw+pw]

        # Pad if extracted patch is smaller than target
        pad_d = self.patch_size[0] - ip.shape[1]
        pad_h = self.patch_size[1] - ip.shape[2]
        pad_w = self.patch_size[2] - ip.shape[3]
        if pad_d > 0 or pad_h > 0 or pad_w > 0:
            ip = np.pad(ip, ((0,0),(0,pad_d),(0,pad_h),(0,pad_w)), mode='constant')
            mp = np.pad(mp, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='constant')

        return ip, mp

    @staticmethod
    def _augment(image: np.ndarray, mask: np.ndarray):
        """Light augmentation: random flips + intensity jitter."""
        for axis in [1, 2, 3]:  # D, H, W (image has C dim at 0)
            if np.random.random() > 0.5:
                image = np.flip(image, axis=axis).copy()
                mask = np.flip(mask, axis=axis - 1).copy()  # mask has no C dim
        # Small intensity noise
        if np.random.random() > 0.5:
            noise = np.random.normal(0, 0.02, image.shape).astype(np.float32)
            image = image + noise
        return image, mask

print("LIDCPretrainingDataset class defined.")

## 6. Sanity-Check: Load a Single Sample

Verify the dataset produces valid tensors and visualise a sample patch.

In [ ]:
# Quick test on one patient
test_ds = LIDCPretrainingDataset(
    patients=train_patients[:1],
    pipeline_config=pipeline_config,
    patch_size=PATCH_SIZE,
    augment=False,
)

sample = test_ds[0]
print(f"Patient:     {sample['patient_id']}")
print(f"Image shape: {sample['image'].shape}  dtype: {sample['image'].dtype}")
print(f"Mask shape:  {sample['mask'].shape}   dtype: {sample['mask'].dtype}")
print(f"Mask unique: {sample['mask'].unique().tolist()}")
print(f"Foreground:  {(sample['mask'] > 0).sum().item()} voxels")

# Visualise middle slices
mid = sample['image'].shape[1] // 2
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(sample['image'][0, mid], cmap='gray')
axes[0].set_title('CT patch (axial)')
axes[1].imshow(sample['mask'][mid], cmap='Reds', vmin=0, vmax=1)
axes[1].set_title('Consensus mask')
axes[2].imshow(sample['image'][0, mid], cmap='gray')
axes[2].imshow(sample['mask'][mid], cmap='Reds', alpha=0.4)
axes[2].set_title('Overlay')
for ax in axes:
    ax.axis('off')
fig.suptitle(f"Sample patch — {sample['patient_id']}", fontsize=12)
fig.tight_layout()
plt.show()

## 7. Pre-extract Patches to SMB Share

Reading raw NIfTI volumes over SMB + full preprocessing each epoch is too slow
(~5 hours per epoch). Instead, we run the heavy I/O **once**: extract 8 patches per
patient and save them as lightweight `.npz` files back to the governed share.

Training then reads these small numpy arrays — **~100× faster per sample**.

In [ ]:
from tqdm.auto import tqdm

train_patch_dir = PATCH_CACHE_DIR / "train"
val_patch_dir   = PATCH_CACHE_DIR / "val"

# Skip extraction if patches already exist
_n_train = len(list(train_patch_dir.glob("*.npz"))) if train_patch_dir.exists() else 0
_n_val   = len(list(val_patch_dir.glob("*.npz")))   if val_patch_dir.exists()   else 0

if _n_train > 0 and _n_val > 0:
    print(f"✓ Patches already extracted — skipping pre-extraction.")
    print(f"  Train: {_n_train} .npz files")
    print(f"  Val:   {_n_val} .npz files")
else:
    def preextract_patches(
        patients: List[Dict],
        output_dir: Path,
        pipeline_config: PipelineConfig,
        patch_size: Tuple[int, int, int],
        patches_per_patient: int,
        label: str = "",
    ):
        """Extract patches from raw NIfTI volumes and save as .npz on the share.

        Skips patients whose patches already exist (resume-safe).
        """
        output_dir.mkdir(parents=True, exist_ok=True)
        ds = LIDCPretrainingDataset(
            patients=patients,
            pipeline_config=pipeline_config,
            patch_size=patch_size,
            augment=False,
        )

        created = 0
        skipped = 0
        failed  = 0

        for idx in tqdm(range(len(ds)), desc=f"Pre-extracting {label} patches"):
            pid = patients[idx]["patient_id"]

            first_missing = None
            for p_idx in range(patches_per_patient):
                npz_path = output_dir / f"{pid}_patch{p_idx:02d}.npz"
                if not npz_path.exists():
                    first_missing = p_idx
                    break

            if first_missing is None:
                skipped += patches_per_patient
                continue

            try:
                rec = patients[idx]
                ct_image = sitk.ReadImage(str(rec["ct_path"]))

                ref_image = ct_image
                merged_arr = None
                for seg_path in rec["seg_paths"]:
                    try:
                        seg = sitk.ReadImage(str(seg_path))
                        if seg.GetDimension() > 3:
                            arr_4d = sitk.GetArrayFromImage(seg)
                            while arr_4d.ndim > 3:
                                arr_4d = arr_4d.max(axis=0)
                            seg = sitk.GetImageFromArray(arr_4d)
                            seg.SetOrigin(ref_image.GetOrigin())
                            seg.SetSpacing(ref_image.GetSpacing())
                            seg.SetDirection(ref_image.GetDirection())
                        if seg.GetSize() != ref_image.GetSize() or \
                           seg.GetSpacing() != ref_image.GetSpacing():
                            seg = sitk.Resample(
                                seg, ref_image, sitk.Transform(),
                                sitk.sitkNearestNeighbor, 0, seg.GetPixelID(),
                            )
                        arr = sitk.GetArrayFromImage(seg)
                        if merged_arr is None:
                            merged_arr = (arr > 0).astype(np.uint8)
                        else:
                            merged_arr = np.maximum(merged_arr, (arr > 0).astype(np.uint8))
                    except Exception:
                        continue

                if merged_arr is None:
                    merged_arr = np.zeros(
                        sitk.GetArrayFromImage(ct_image).shape, dtype=np.uint8
                    )

                mask_image = sitk.GetImageFromArray(merged_arr.astype(np.int32))
                mask_image.CopyInformation(ref_image)

                image_arr, mask_arr = ds.preprocessor.preprocess_for_deep_learning(
                    ct_image, mask_image
                )

                for p_idx in range(first_missing, patches_per_patient):
                    npz_path = output_dir / f"{pid}_patch{p_idx:02d}.npz"
                    if npz_path.exists():
                        skipped += 1
                        continue
                    img_patch, msk_patch = ds._extract_patch(image_arr, mask_arr)
                    np.savez_compressed(
                        npz_path,
                        image=img_patch.astype(np.float16),
                        mask=msk_patch.astype(np.uint8),
                    )
                    created += 1

            except Exception as e:
                logger.warning(f"Failed to extract patches for {pid}: {e}")
                failed += patches_per_patient
                continue

        return created, skipped, failed

    print(f"Patch cache directory: {PATCH_CACHE_DIR}")
    print(f"Extracting {PATCHES_PER_PATIENT} patches per patient...\n")

    t0 = time.time()
    cr_t, sk_t, fa_t = preextract_patches(
        train_patients, train_patch_dir, pipeline_config,
        PATCH_SIZE, PATCHES_PER_PATIENT, label="train",
    )
    cr_v, sk_v, fa_v = preextract_patches(
        val_patients, val_patch_dir, pipeline_config,
        PATCH_SIZE, PATCHES_PER_PATIENT, label="val",
    )
    elapsed = time.time() - t0

    print(f"\n✓ Pre-extraction complete in {elapsed/60:.1f} min")
    print(f"  Train: {cr_t} created, {sk_t} skipped, {fa_t} failed")
    print(f"  Val:   {cr_v} created, {sk_v} skipped, {fa_v} failed")
    print(f"  Total .npz files: {cr_t + sk_t + cr_v + sk_v}")

## 8. Cached Patch Dataset

Fast `Dataset` that reads pre-extracted `.npz` files instead of raw NIfTIs.
Augmentation (random flips + intensity jitter) is still applied on-the-fly.

In [ ]:
class CachedPatchDataset(Dataset):
    """Reads pre-extracted .npz patches — no SimpleITK, no raw NIfTI I/O."""

    def __init__(self, patch_dir: Path, augment: bool = True):
        self.patch_dir = patch_dir
        self.augment = augment
        self.npz_files = sorted(patch_dir.glob("*.npz"))
        if len(self.npz_files) == 0:
            raise FileNotFoundError(f"No .npz files found in {patch_dir}")

    def __len__(self) -> int:
        return len(self.npz_files)

    def __getitem__(self, idx: int) -> Dict:
        data = np.load(self.npz_files[idx])
        image = data["image"].astype(np.float32)  # (1, D, H, W) — stored as float16
        mask  = data["mask"]                       # (D, H, W) uint8

        if self.augment:
            image, mask = self._augment(image, mask)

        # Extract patient_id from filename: LIDC-IDRI-0001_patch03.npz
        pid = self.npz_files[idx].stem.rsplit("_patch", 1)[0]

        return {
            "image": torch.from_numpy(image).float(),
            "mask":  torch.from_numpy(mask).long(),
            "patient_id": pid,
        }

    @staticmethod
    def _augment(image: np.ndarray, mask: np.ndarray):
        """Random flips + intensity jitter (same as LIDCPretrainingDataset)."""
        for axis in [1, 2, 3]:
            if np.random.random() > 0.5:
                image = np.flip(image, axis=axis).copy()
                mask = np.flip(mask, axis=axis - 1).copy()
        if np.random.random() > 0.5:
            noise = np.random.normal(0, 0.02, image.shape).astype(np.float32)
            image = image + noise
        return image, mask


print(f"CachedPatchDataset class defined.")
print(f"  Train patches: {len(list((PATCH_CACHE_DIR / 'train').glob('*.npz')))} .npz files")
print(f"  Val patches:   {len(list((PATCH_CACHE_DIR / 'val').glob('*.npz')))} .npz files")

In [ ]:
train_ds = CachedPatchDataset(train_patch_dir, augment=True)
val_ds   = CachedPatchDataset(val_patch_dir,   augment=False)

# num_workers=0 avoids macOS spawn-context hang on SMB-hosted .npz files
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)

print(f"Train samples: {len(train_ds)},  batches/epoch: {len(train_loader)}")
print(f"Val samples:   {len(val_ds)},  batches/epoch: {len(val_loader)}")

## 9. Model Setup: ResNet3D-18 Encoder + Segmentation Decoder

In [ ]:
# --- Encoder: ResNet3D-18 --------------------------------------------------
encoder = create_encoder(
    encoder_type=ENCODER_TYPE,
    latent_dim=LATENT_DIM,
    in_channels=1,
)

# --- Decoder: lightweight segmentation head --------------------------------
# ResNet-18 uses BasicBlock (expansion=1), so final feature map = 512 channels
encoder_channels = 512 * BasicBlock.expansion  # = 512
decoder = SegmentationDecoder(
    encoder_channels=encoder_channels,
    num_classes=2,         # background + nodule
    base_channels=64,
)

encoder = encoder.to(DEVICE)
decoder = decoder.to(DEVICE)

total_enc = sum(p.numel() for p in encoder.parameters())
total_dec = sum(p.numel() for p in decoder.parameters())
print(f"Encoder params: {total_enc:,}")
print(f"Decoder params: {total_dec:,}")
print(f"Total:          {total_enc + total_dec:,}")

## 10. Training Loop

Train using **combined Dice + BCE loss** with early stopping.
Checkpoints are saved to `checkpoints/lidc_pretrain/`.

In [ ]:
def dice_score(pred: torch.Tensor, target: torch.Tensor, smooth: float = 1e-6) -> float:
    """Compute Dice coefficient for binary segmentation."""
    pred_bin = pred.argmax(dim=1)  # (B, D, H, W)
    intersection = (pred_bin * target).sum().float()
    union = pred_bin.sum().float() + target.sum().float()
    return ((2.0 * intersection + smooth) / (union + smooth)).item()


def run_epoch(encoder, decoder, loader, optimizer=None, device=DEVICE, max_batches=None):
    """Run one train or validation epoch. Pass optimizer=None for val."""
    is_train = optimizer is not None
    encoder.train(is_train)
    decoder.train(is_train)

    total_loss = 0.0
    total_dice = 0.0
    n_batches  = 0

    ctx = torch.no_grad if not is_train else torch.enable_grad
    with ctx():
        for i, batch in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break

            images = batch["image"].to(device)
            masks  = batch["mask"].to(device)

            # Encoder → feature maps (not latent) → Decoder
            features = encoder.forward_features(images)
            logits   = decoder(features)

            # Match spatial dims
            if logits.shape[2:] != masks.shape[1:]:
                logits = F.interpolate(
                    logits, size=masks.shape[1:],
                    mode="trilinear", align_corners=False,
                )

            loss = F.cross_entropy(logits, masks)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_dice += dice_score(logits, masks)
            n_batches  += 1

    return total_loss / max(n_batches, 1), total_dice / max(n_batches, 1)

print("Training functions defined.")

In [ ]:
# --- Optimizer & scheduler -------------------------------------------------
params = list(encoder.parameters()) + list(decoder.parameters())
optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# --- Mini-epoch config -----------------------------------------------------
# ~200 patients × 8 patches ÷ batch_size 4 = 400 batches per pass
# Loader is recreated each pass to reshuffle patches
BATCHES_PER_PASS = 400
PATIENCE = 10

history = []
best_val_loss = float("inf")
patience_counter = 0

print(f"Starting training — {NUM_EPOCHS} passes, {BATCHES_PER_PASS} batches/pass, patience={PATIENCE}")
print(f"{'Pass':>5} | {'Train Loss':>10} | {'Train Dice':>10} | "
      f"{'Val Loss':>10} | {'Val Dice':>10} | {'LR':>10} | Time")
print("-" * 80)

for epoch in range(NUM_EPOCHS):
    t0 = time.time()

    # Recreate loader each pass to reshuffle
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=False)

    train_loss, train_dice = run_epoch(encoder, decoder, train_loader, optimizer,
                                       max_batches=BATCHES_PER_PASS)
    val_loss,   val_dice   = run_epoch(encoder, decoder, val_loader, None)

    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]
    elapsed = time.time() - t0

    history.append({
        "pass": epoch + 1,
        "train_loss": train_loss, "train_dice": train_dice,
        "val_loss":   val_loss,   "val_dice":   val_dice,
        "lr": lr_now,
    })

    print(f"{epoch+1:5d} | {train_loss:10.4f} | {train_dice:10.4f} | "
          f"{val_loss:10.4f} | {val_dice:10.4f} | {lr_now:10.2e} | {elapsed:.0f}s")

    # Checkpoint best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            "pass": epoch + 1,
            "encoder_state_dict": encoder.state_dict(),
            "decoder_state_dict": decoder.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "config": {
                "encoder_type": ENCODER_TYPE,
                "latent_dim":   LATENT_DIM,
                "patch_size":   PATCH_SIZE,
            }
        }, CHECKPOINT_DIR / "encoder_best.pt")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at pass {epoch+1} "
                  f"(no improvement for {PATIENCE} passes)")
            break

# Save final checkpoint
torch.save({
    "pass": epoch + 1,
    "encoder_state_dict": encoder.state_dict(),
    "decoder_state_dict": decoder.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "val_loss": val_loss,
    "config": {
        "encoder_type": ENCODER_TYPE,
        "latent_dim":   LATENT_DIM,
        "patch_size":   PATCH_SIZE,
    }
}, CHECKPOINT_DIR / "encoder_final.pt")

print(f"\n✓ Training complete.  Best val loss: {best_val_loss:.4f}")
print(f"  Checkpoints saved to: {CHECKPOINT_DIR}")

In [ ]:
print(f"Completed epochs: {len(history)}")
print(f"Current epoch variable: {epoch}")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Patience counter: {patience_counter}")
if history:
    for i, h in enumerate(history):
        print(f"  Epoch {i+1}: train_loss={h['train_loss']:.4f}, val_loss={h['val_loss']:.4f}, "
              f"train_dice={h['train_dice']:.4f}, val_dice={h['val_dice']:.4f}")
else:
    print("No epochs completed yet.")

## 11. Training Curves

In [ ]:
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Loss
axes[0].plot(hist_df["pass"], hist_df["train_loss"], label="Train")
axes[0].plot(hist_df["pass"], hist_df["val_loss"],   label="Val")
axes[0].set_xlabel("Pass"); axes[0].set_ylabel("CE Loss")
axes[0].set_title("Loss"); axes[0].legend()
axes[0].grid(True, alpha=0.3)

# (b) Dice
axes[1].plot(hist_df["pass"], hist_df["train_dice"], label="Train")
axes[1].plot(hist_df["pass"], hist_df["val_dice"],   label="Val")
axes[1].set_xlabel("Pass"); axes[1].set_ylabel("Dice")
axes[1].set_title("Dice Score"); axes[1].legend()
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

# (c) Learning rate
axes[2].plot(hist_df["pass"], hist_df["lr"], color="tab:orange")
axes[2].set_xlabel("Pass"); axes[2].set_ylabel("LR")
axes[2].set_title("Learning Rate Schedule")
axes[2].grid(True, alpha=0.3)

fig.suptitle("LIDC Segmentation Pretraining", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 12. Qualitative Validation — Predict on Example Patches

In [ ]:
# Load best checkpoint
ckpt = torch.load(CHECKPOINT_DIR / "encoder_best.pt", map_location=DEVICE)
encoder.load_state_dict(ckpt["encoder_state_dict"])
decoder.load_state_dict(ckpt["decoder_state_dict"])
encoder.eval()
decoder.eval()

# Predict on 4 validation samples
n_show = min(4, len(val_ds))
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

for i in range(n_show):
    sample = val_ds[i]
    img = sample["image"].unsqueeze(0).to(DEVICE)
    msk = sample["mask"]
    mid = img.shape[2] // 2

    with torch.no_grad():
        feat = encoder.forward_features(img)
        logits = decoder(feat)
        if logits.shape[2:] != msk.shape:
            logits = F.interpolate(logits, size=msk.shape,
                                   mode="trilinear", align_corners=False)
        pred = logits.argmax(dim=1).squeeze(0).cpu()

    axes[i, 0].imshow(sample["image"][0, mid].numpy(), cmap="gray")
    axes[i, 0].set_title(f"{sample['patient_id']} — CT")
    axes[i, 1].imshow(msk[mid].numpy(), cmap="Reds", vmin=0, vmax=1)
    axes[i, 1].set_title("Ground truth")
    axes[i, 2].imshow(pred[mid].numpy(), cmap="Reds", vmin=0, vmax=1)
    axes[i, 2].set_title("Prediction")
    # Overlay
    axes[i, 3].imshow(sample["image"][0, mid].numpy(), cmap="gray")
    axes[i, 3].imshow(pred[mid].numpy(), cmap="Reds", alpha=0.4)
    axes[i, 3].set_title("Overlay")
    for ax in axes[i]:
        ax.axis("off")

fig.suptitle("Validation Predictions (best checkpoint)", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 13. Extract Deep Features for NSCLC-Radiomics Cohort

Use the pretrained encoder to generate 512-dim feature vectors for each
NSCLC-Radiomics patient. These will be combined with handcrafted radiomic
features in the downstream hybrid classifier.

In [ ]:
from tqdm.auto import tqdm

NSCLC_ROOT = Path("/Volumes/AIR-NSCLC/nifti_output/NSCLC-Radiomics")
assert NSCLC_ROOT.exists(), f"NSCLC root not found: {NSCLC_ROOT}"

# Discover NSCLC patients
nsclc_dirs = sorted([d for d in NSCLC_ROOT.iterdir() if d.is_dir()])
print(f"NSCLC-Radiomics patients: {len(nsclc_dirs)}")
print(f"  First: {nsclc_dirs[0].name}  Last: {nsclc_dirs[-1].name}")

In [ ]:
def load_nsclc_mask(seg_path: Path, reference_image: sitk.Image) -> sitk.Image:
    """Load NSCLC-Radiomics segmentation, collapsing 4D DICOM SEG to 3D then resampling to CT grid."""
    seg = sitk.ReadImage(str(seg_path))
    if seg.GetDimension() > 3:
        arr = sitk.GetArrayFromImage(seg)
        while arr.ndim > 3:
            arr = arr.max(axis=0)
        seg = sitk.GetImageFromArray(arr)
        seg.SetOrigin(reference_image.GetOrigin())
        seg.SetSpacing(reference_image.GetSpacing())
        seg.SetDirection(reference_image.GetDirection())
    # Resample onto CT grid — guarantees identical size/spacing for preprocessor
    seg = sitk.Resample(
        sitk.Cast(seg, sitk.sitkInt32),
        reference_image,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        sitk.sitkInt32,
    )
    return seg


def extract_roi_crop(image: np.ndarray, mask: np.ndarray,
                     patch_size: Tuple[int, int, int] = (64, 64, 64)) -> np.ndarray:
    """Crop patch centred on tumour centroid — mirrors training patch distribution."""
    _, d, h, w = image.shape
    pd, ph, pw = patch_size

    fg = np.where(mask > 0)
    if len(fg[0]) > 0:
        cd = int(fg[0].mean())
        ch = int(fg[1].mean())
        cw = int(fg[2].mean())
    else:
        cd, ch, cw = d // 2, h // 2, w // 2

    sd = max(0, min(cd - pd // 2, d - pd))
    sh = max(0, min(ch - ph // 2, h - ph))
    sw = max(0, min(cw - pw // 2, w - pw))

    crop = image[:, sd:sd+pd, sh:sh+ph, sw:sw+pw]

    pad_d = pd - crop.shape[1]
    pad_h = ph - crop.shape[2]
    pad_w = pw - crop.shape[3]
    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        crop = np.pad(crop, ((0,0),(0,pad_d),(0,pad_h),(0,pad_w)), mode='constant')

    return crop


# --- Load best encoder (frozen) -------------------------------------------
ckpt = torch.load(CHECKPOINT_DIR / "encoder_best.pt", map_location=DEVICE)
encoder.load_state_dict(ckpt["encoder_state_dict"])
encoder.eval()

preprocessor = ImagePreprocessor(pipeline_config)

# --- Extract 512-dim deep features per NSCLC patient -----------------------
deep_features = []
extract_log   = []

for patient_dir in tqdm(nsclc_dirs, desc="Deep feature extraction"):
    pid = patient_dir.name
    ct_path = patient_dir / "ct.nii.gz"
    seg_files = sorted(patient_dir.glob("Segmentation*.nii.gz"))

    if not ct_path.exists() or len(seg_files) == 0:
        extract_log.append({"patient_id": pid, "status": "missing_files"})
        continue

    try:
        ct_image   = sitk.ReadImage(str(ct_path))
        mask_image = load_nsclc_mask(seg_files[0], ct_image)

        img_arr, mask_arr = preprocessor.preprocess_for_deep_learning(
            ct_image, mask_image
        )

        roi = extract_roi_crop(img_arr, mask_arr, patch_size=PATCH_SIZE)

        img_t = torch.from_numpy(roi).unsqueeze(0).float().to(DEVICE)

        with torch.no_grad():
            latent    = encoder(img_t)
            latent_np = latent.cpu().numpy().flatten()

        feat_dict = {"patient_id": pid}
        for i, v in enumerate(latent_np):
            feat_dict[f"deep_feat_{i:04d}"] = float(v)

        deep_features.append(feat_dict)
        extract_log.append({"patient_id": pid, "status": "success"})

    except Exception as e:
        logger.error(f"{pid}: {e}")
        extract_log.append({"patient_id": pid, "status": "failed", "error": str(e)})

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

deep_df = pd.DataFrame(deep_features)
log_df  = pd.DataFrame(extract_log)

n_ok = (log_df["status"] == "success").sum()
print(f"\n✓ Deep features extracted: {n_ok}/{len(nsclc_dirs)} patients")
print(f"  Feature vector shape: {deep_df.shape}")

## 14. Visualise Deep Feature Space (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

feat_cols = [c for c in deep_df.columns if c.startswith("deep_feat_")]
X_deep = deep_df[feat_cols].values

# t-SNE embedding
perplexity = min(30, len(X_deep) - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
emb = tsne.fit_transform(X_deep)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(emb[:, 0], emb[:, 1], s=20, alpha=0.7, edgecolors="white", linewidths=0.3)
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_title(f"NSCLC-Radiomics Deep Feature Embedding (n={len(emb)})")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 15. Save Deep Features to Disk

Save as CSV so downstream notebooks can `pd.read_csv()` without re-running feature extraction.

In [ ]:
output_csv = Path.cwd() / "nsclc_deep_features.csv"
deep_df.to_csv(output_csv, index=False)
print(f"✓ Deep features saved: {output_csv}")
print(f"  Shape: {deep_df.shape}")
print(f"  Columns: patient_id + {len(feat_cols)} deep features")

## 16. Summary

### What we did
1. **Discovered** 872 LIDC-IDRI patients with segmentation masks on the AIR-NSCLC share
2. **Pre-extracted** patches as `.npz` files to eliminate per-epoch SMB I/O bottleneck
3. **Pretrained** a ResNet3D-18 encoder on binary segmentation (background vs nodule)
4. **Extracted** 512-dim deep feature vectors for the NSCLC-Radiomics cohort

### Next Steps
1. Load `nsclc_deep_features.csv` in the hybrid classification notebook
2. Combine with handcrafted PyRadiomics features (ComBat-harmonised)
3. Train response/outcome classifiers on the fused feature set